# Schema Creation — SBA Lending Project
# **CPSC 5071** | Bruna & Jack
#
# This notebook creates the SQLite database schema for the SBA lending equity analysis.
# Four tables: `industry_sector`, `demographic_profile`, `economic_indicator`, `loan`.


In [ ]:
import sqlite3

# Create the database
conn = sqlite3.connect('sba_lending.db')
cursor = conn.cursor()

# Enable foreign keys
cursor.execute("PRAGMA foreign_keys = ON;")

# ============================================================
# Table 1: industry_sector (reference/lookup table)
# ============================================================
cursor.execute("""
CREATE TABLE IF NOT EXISTS industry_sector (
    naics_2digit    TEXT PRIMARY KEY,
    sector_name     TEXT NOT NULL
);
""")

# ============================================================
# Table 2: demographic_profile (from Census ABS API)
# One row per NAICS sector × State × Year combination
# ============================================================
cursor.execute("""
CREATE TABLE IF NOT EXISTS demographic_profile (
    naics_2digit        TEXT NOT NULL,
    state               TEXT NOT NULL,
    data_year           INTEGER NOT NULL,
    total_firms         INTEGER,
    pct_women_owned     REAL,
    pct_minority_owned  REAL,
    pct_veteran_owned   REAL,
    avg_receipts        REAL,
    PRIMARY KEY (naics_2digit, state, data_year),
    FOREIGN KEY (naics_2digit) REFERENCES industry_sector(naics_2digit)
);
""")

# ============================================================
# Table 3: economic_indicator (from FRED API)
# One row per year × month
# ============================================================
cursor.execute("""
CREATE TABLE IF NOT EXISTS economic_indicator (
    year            INTEGER NOT NULL,
    month           INTEGER NOT NULL,
    fed_funds_rate  REAL,
    prime_rate      REAL,
    cpi             REAL,
    unemployment    REAL,
    PRIMARY KEY (year, month)
);
""")

# ============================================================
# Table 4: loan (core table — one row per SBA loan)
# ============================================================
cursor.execute("""
CREATE TABLE IF NOT EXISTS loan (
    loan_id             INTEGER PRIMARY KEY,
    business_name       TEXT,
    city                TEXT,
    state               TEXT,
    zip                 INTEGER,
    bank_name           TEXT,
    bank_state          TEXT,
    naics               INTEGER,
    naics_2digit        TEXT,
    approval_date       DATE,
    approval_year       INTEGER,
    approval_month      INTEGER,
    term_months         INTEGER,
    num_employees       INTEGER,
    new_or_existing     INTEGER,
    jobs_created        INTEGER,
    jobs_retained       INTEGER,
    franchise_code      INTEGER,
    urban_rural         INTEGER,
    revolving_line      TEXT,
    low_doc             TEXT,
    disbursement_date   DATE,
    disbursement_gross  REAL,
    balance_gross       REAL,
    gross_approved      REAL,
    sba_approved        REAL,
    chgoff_date         DATE,
    chgoff_principal    REAL,
    is_default          INTEGER NOT NULL,
    FOREIGN KEY (naics_2digit) REFERENCES industry_sector(naics_2digit),
    FOREIGN KEY (naics_2digit, state, approval_year) REFERENCES demographic_profile(naics_2digit, state, data_year),
    FOREIGN KEY (approval_year, approval_month) REFERENCES economic_indicator(year, month)
);
""")

conn.commit()

# Verify tables were created
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print("Tables created:", [row[0] for row in cursor.fetchall()])

# Check schema for each table
for table in ['industry_sector', 'demographic_profile', 'economic_indicator', 'loan']:
    print(f"\n{table.upper()}:")
    cursor.execute(f"PRAGMA table_info({table});")
    for col in cursor.fetchall():
        print(f"  {col[1]:<25} {col[2]:<10} {'PK' if col[5] else ''} {'NOT NULL' if col[3] else ''}")

# Verify foreign key relationships
print("FOREIGN KEYS ON LOAN TABLE:")
cursor.execute("PRAGMA foreign_key_list(loan);")
for fk in cursor.fetchall():
    print(f"  {fk[3]} → {fk[2]}.{fk[4]}")

conn.close()

Tables created: ['industry_sector', 'demographic_profile', 'economic_indicator', 'loan']

INDUSTRY_SECTOR:
  naics_2digit              TEXT       PK 
  sector_name               TEXT        NOT NULL

DEMOGRAPHIC_PROFILE:
  naics_2digit              TEXT       PK NOT NULL
  state                     TEXT       PK NOT NULL
  data_year                 INTEGER    PK NOT NULL
  total_firms               INTEGER     
  pct_women_owned           REAL        
  pct_minority_owned        REAL        
  pct_veteran_owned         REAL        
  avg_receipts              REAL        

ECONOMIC_INDICATOR:
  year                      INTEGER    PK NOT NULL
  month                     INTEGER    PK NOT NULL
  fed_funds_rate            REAL        
  prime_rate                REAL        
  cpi                       REAL        
  unemployment              REAL        

LOAN:
  loan_id                   INTEGER    PK 
  business_name             TEXT        
  city                      TEXT        
  